# Figure 5: Distribution-Level Structure Recovery

- Figure / panels: `Fig5a`, `Fig5b`, `Fig5c`, `Fig5d`
- Inputs: `benchmark_summary_all.csv`, split-level `metrics*.csv`, payload `pkl`, raw dataset `h5ad`
- Outputs: `artifacts/paper_figures/main/Fig5_DistributionRecovery/*`
- Role: Main-text distribution-level recovery module
- Notes: `Fig5a` uses a boxplot for Wasserstein distance; `Fig5d` is a single-condition violin plot


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'scripts').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from scripts.common.paper_plot_style import apply_gears_paper_style, style_axis as paper_style_axis
apply_gears_paper_style(font_scale=1.0)

DATASET_ORDER = ['Adamson', 'Norman', 'Dixit']
MODEL_ORDER = ['trishift_nearest', 'gears', 'biolord', 'genepert', 'scgpt']
MODEL_LABELS = {'trishift_nearest':'TriShift','gears':'GEARS','biolord':'biolord','genepert':'GenePert','scgpt':'scGPT','Truth':'Truth'}
MODEL_COLORS = {'TriShift':'#9FD9D3','biolord':'#F0806A','GEARS':'#F2B56B','GenePert':'#87A8D8','scGPT':'#DDD3C8','Truth':'#7F7F7F'}
OUT_ROOT = repo_root / 'artifacts' / 'paper_figures' / 'main' / 'Fig5_DistributionRecovery'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

def _style_axis(ax, grid_axis="y"):
    paper_style_axis(ax, grid_axis=grid_axis)
    ax.set_axisbelow(True)

def _shrink_bars(ax, scale=0.9):
    for patch in ax.patches:
        width = patch.get_width()
        if width <= 0:
            continue
        new_width = width * scale
        patch.set_x(patch.get_x() + (width - new_width) / 2)
        patch.set_width(new_width)


In [ ]:
summary_path = repo_root / 'artifacts' / 'paper_figures' / 'main' / 'Fig2_MultiDatasetBenchmark' / 'benchmark_summary_all.csv'
summary_df = pd.read_csv(summary_path)
summary_df = summary_df[summary_df['model_name'].isin(MODEL_ORDER)].copy()
summary_df['dataset_label'] = pd.Categorical(summary_df['dataset_label'], DATASET_ORDER, ordered=True)
summary_df['display_name'] = summary_df['model_name'].map(MODEL_LABELS)

detail_rows = []
for row in summary_df.itertuples(index=False):
    metrics_path = Path(row.metrics_path)
    if not metrics_path.exists():
        continue
    metrics_df = pd.read_csv(metrics_path)
    if str(row.dataset).strip().lower() == 'norman' and 'subgroup' in metrics_df.columns:
        metrics_df = metrics_df[metrics_df['subgroup'].astype(str).str.lower() == 'single'].copy()
    if metrics_df.empty:
        continue
    metric_col = 'scpram_wasserstein_degs_sum'
    if metric_col not in metrics_df.columns:
        continue
    keep = metrics_df[['condition', metric_col]].copy()
    keep['dataset'] = row.dataset
    keep['dataset_label'] = row.dataset_label
    keep['model_name'] = row.model_name
    keep['display_name'] = row.display_name
    detail_rows.append(keep)

detail_df = pd.concat(detail_rows, ignore_index=True) if detail_rows else pd.DataFrame()
detail_df['dataset_label'] = pd.Categorical(detail_df['dataset_label'], DATASET_ORDER, ordered=True)

display(summary_df.head())
display(detail_df.head())


In [ ]:
metrics_to_plot = [
    ('mean_scpram_wasserstein_degs_sum', 'Wasserstein distance', 'fig5a_wasserstein_boxplot.png'),
    ('mean_scpram_r2_degs_mean_mean', r'$R^2$ of mean', 'fig5b_r2_mean_barplot.png'),
    ('mean_scpram_r2_degs_var_mean', r'$R^2$ of variance', 'fig5c_r2_variance_barplot.png'),
]

for metric_col, title, out_name in metrics_to_plot:
    if metric_col == 'mean_scpram_wasserstein_degs_sum':
        plot_df = detail_df.dropna(subset=['scpram_wasserstein_degs_sum']).copy()
        if plot_df.empty:
            continue
        present_hues = [MODEL_LABELS[m] for m in MODEL_ORDER if MODEL_LABELS[m] in set(plot_df['display_name'].tolist())]
        fig, ax = plt.subplots(figsize=(7.2, 4.6), dpi=220)
        sns.boxplot(
            data=plot_df,
            x='dataset_label',
            y='scpram_wasserstein_degs_sum',
            hue='display_name',
            order=DATASET_ORDER,
            hue_order=present_hues,
            palette=MODEL_COLORS,
            ax=ax,
            linewidth=0.9,
            fliersize=1.8,
            width=0.72,
        )
        for patch in ax.artists:
            patch.set_edgecolor('black')
            patch.set_linewidth(0.8)
        for line in ax.lines:
            line.set_color('black')
            line.set_linewidth(0.8)
        ax.set_xlabel('')
        ax.set_ylabel(title)
        ax.set_title(title, pad=10)
        ax.legend(title='', frameon=False, ncol=min(len(present_hues), 5), loc='upper center', bbox_to_anchor=(0.5, 1.12), borderaxespad=0.0)
        _style_axis(ax, grid_axis='y')
        fig.tight_layout(rect=[0, 0, 1, 0.94])
        fig.savefig(OUT_ROOT / out_name, bbox_inches='tight')
        plot_df.to_csv(OUT_ROOT / 'fig5a_wasserstein_boxplot_values.csv', index=False)
        plt.close(fig)
    else:
        plot_df = summary_df.dropna(subset=[metric_col]).copy()
        if plot_df.empty:
            continue
        present_hues = [MODEL_LABELS[m] for m in MODEL_ORDER if MODEL_LABELS[m] in set(plot_df['display_name'].tolist())]
        fig, ax = plt.subplots(figsize=(6.4, 4.6), dpi=220)
        sns.barplot(
            data=plot_df,
            x='dataset_label',
            y=metric_col,
            hue='display_name',
            order=DATASET_ORDER,
            hue_order=present_hues,
            palette=MODEL_COLORS,
            errorbar=None,
            ax=ax,
        )
        _shrink_bars(ax, scale=0.96)
        for patch in ax.patches:
            patch.set_edgecolor('black')
            patch.set_linewidth(0.6)
        ax.set_xlabel('')
        ax.set_ylabel(title)
        ax.set_title(title, pad=10)
        ax.tick_params(axis='x', rotation=28)
        ax.legend(title='', frameon=False, ncol=min(len(present_hues), 5), loc='upper center', bbox_to_anchor=(0.5, 1.12), borderaxespad=0.0)
        _style_axis(ax, grid_axis='y')
        fig.tight_layout(rect=[0, 0, 1, 0.94])
        fig.savefig(OUT_ROOT / out_name, bbox_inches='tight')
        plt.close(fig)


## Fig5d: Representative violin case

- Condition: `SPCS2+ctrl` (Adamson, split 1)
- Gene: `DDIT4`
- Rationale: strong distribution-level recovery with clear mean/variance separation between control and perturbed cells


In [ ]:
from scripts.trishift.analysis._result_adapter import load_payload_item

FIG5D_SPEC = {
    'dataset': 'adamson',
    'split_id': 1,
    'condition': 'SPCS2+ctrl',
    'gene': 'DDIT4',
    'models': ['trishift_nearest', 'genepert', 'scgpt', 'biolord'],
}

violin_palette = {
    'Control': '#CFCFCF',
    'Perturbed': '#7F7F7F',
    'TriShift': MODEL_COLORS['TriShift'],
    'GenePert': MODEL_COLORS['GenePert'],
    'scGPT': MODEL_COLORS['scGPT'],
    'biolord': MODEL_COLORS['biolord'],
}

base_item = load_payload_item(
    dataset=FIG5D_SPEC['dataset'],
    model_name='trishift_nearest',
    split_id=FIG5D_SPEC['split_id'],
    condition=FIG5D_SPEC['condition'],
)
gene_names = np.asarray(base_item['gene_name_full']).astype(str)
if FIG5D_SPEC['gene'] not in set(gene_names.tolist()):
    raise KeyError(f"Gene not found for Fig5d: {FIG5D_SPEC['gene']}")
gene_idx = int(np.where(gene_names == FIG5D_SPEC['gene'])[0][0])

rows = []
ctrl_vals = np.asarray(base_item['Ctrl_full'], dtype=float)[:, gene_idx]
truth_vals = np.asarray(base_item['Truth_full'], dtype=float)[:, gene_idx]
for value in ctrl_vals:
    rows.append({'group': 'Control', 'expression': float(value)})
for value in truth_vals:
    rows.append({'group': 'Perturbed', 'expression': float(value)})

for model_name in FIG5D_SPEC['models']:
    item = load_payload_item(
        dataset=FIG5D_SPEC['dataset'],
        model_name=model_name,
        split_id=FIG5D_SPEC['split_id'],
        condition=FIG5D_SPEC['condition'],
    )
    model_gene_names = np.asarray(item['gene_name_full']).astype(str)
    model_gene_idx = int(np.where(model_gene_names == FIG5D_SPEC['gene'])[0][0])
    pred_vals = np.asarray(item['Pred_full'], dtype=float)[:, model_gene_idx]
    group_label = MODEL_LABELS[model_name]
    for value in pred_vals:
        rows.append({'group': group_label, 'expression': float(value)})

violin_df = pd.DataFrame(rows)
order = ['Control', 'Perturbed', 'TriShift', 'GenePert', 'scGPT', 'biolord']
fig, ax = plt.subplots(figsize=(6.6, 4.5), dpi=240)
sns.violinplot(
    data=violin_df,
    x='group',
    y='expression',
    hue='group',
    order=order,
    hue_order=order,
    palette=violin_palette,
    inner='quartile',
    cut=0,
    linewidth=0.8,
    dodge=False,
    legend=False,
    width=0.82,
    ax=ax,
)
for coll in ax.collections:
    try:
        coll.set_edgecolor('black')
        coll.set_linewidth(0.8)
    except Exception:
        pass
ax.set_xlabel('')
ax.set_ylabel(f"{FIG5D_SPEC['gene']} expression")
ax.set_title(f"{FIG5D_SPEC['gene']} | {FIG5D_SPEC['condition']}", pad=10)
ax.tick_params(axis='x', labelrotation=28)
_style_axis(ax, grid_axis='y')
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(OUT_ROOT / 'fig5d_violin_case.png', bbox_inches='tight')
violin_df.to_csv(OUT_ROOT / 'fig5d_violin_case_values.csv', index=False)
print(FIG5D_SPEC)
print(violin_df.groupby('group')['expression'].agg(['mean', 'var', 'count']).round(4))
plt.close(fig)
